<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'><a href="https://www.kaggle.com/datasets/eliasdabbas/web-server-access-logs/data">Analyse exploratoire des logs d’un serveur Web</a></div>
     </th>
     <th></th>
 </tr>
<tr>                                                                                   
     <th><img src="https://raw.githubusercontent.com/rbizoi/dockerBigDataBI/refs/heads/master/images/log_web_server.png" width="1024"></th>
 </tr>    
</table>

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Introduction</div></b>
## <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Import libriries </div></b>

In [1]:
import pandas as pd, os
from outils.web_log_dataframe import build_dataframe
from pathlib import Path

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture des données</div></b>

<table>
    <tr> 
        <th>
             <img src="https://raw.githubusercontent.com/rbizoi/dockerBigDataBI/refs/heads/master/images/log_files/log01.png" width="512">
        </th>
        <th>
           <img src="https://raw.githubusercontent.com/rbizoi/dockerBigDataBI/refs/heads/master/images/log_files/maxmind.png" width="512"></th>
        </tr>
</table>

<div><a href="https://sourceforge.net/projects/geolite-mmdb.mirror">MaxMind GeoIP2 GeoLite2 Country, City, and ASN databases </a></div>

In [2]:
# Liste minimale des colonnes que le script principal doit produire.
# Le test échoue si une de ces colonnes disparaît à la suite d’une modification.
EXPECTED_COLUMNS = {
    "ip", "datetime", "method", "url", "protocol", "status", "bytes",
    "referer", "user_agent", "ip_version", "ip_is_private", "ip_is_global",
    "url_path", "url_query", "referer_scheme", "referer_host", "referer_domain",
    "referer_path", "referer_query", "referer_type", "search_engine", "search_query",
    "social_network", "is_search_engine", "is_social", "is_direct",
    "browser", "browser_version", "os", "os_version", "device_type",
    "device_family", "device_brand", "device_model", "is_mobile", "is_tablet",
    "is_pc", "is_bot", "bot_name", "client_type",
    "geo_country", "geo_country_code", "geo_city", "geo_latitude", "geo_longitude",
}

def check(name, condition, details=""):
    """Affiche un contrôle sous la forme [OK]/[ECHEC] et retourne un booléen."""
    ok = bool(condition)
    print(f"[{'OK' if ok else 'ECHEC'}] {name}" + (f" — {details}" if details else ""))
    return ok

In [3]:
%%time
# path = Path(LOG_FILE)
repertoire="/opt/spark/data/log_web_access"
results = []
# results.append(check("fichier présent", path.exists(), str(path)))
donnees = pd.DataFrame()

# ------------------------------------------------------------------
# 2. Test d’intégration : construction réelle du DataFrame.
# ------------------------------------------------------------------
try:
    for fichier in os.listdir(repertoire)[:10]:
        df = build_dataframe(log_file=os.path.join(repertoire,fichier), site_domain="zanbil.ir", geoip_db="outils/GeoLite2-City.mmdb")
        donnees = pd.concat([donnees,df],ignore_index=True)
    results.append(check("build_dataframe s'exécute sans exception", True))
except Exception as exc:
    check("build_dataframe s'exécute sans exception", False, repr(exc)); 


[OK] build_dataframe s'exécute sans exception
CPU times: user 6.98 s, sys: 634 ms, total: 7.62 s
Wall time: 10.4 s


In [4]:
donnees.shape

(499997, 55)

In [5]:
!mkdir /opt/spark/notebooks/log_web_access

In [6]:
donnees.to_parquet('/opt/spark/notebooks/log_web_access/Web.Server.Access10.gzip',compression='gzip', engine='pyarrow')

In [7]:
donnees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499997 entries, 0 to 499996
Data columns (total 55 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   ip                   499997 non-null  object             
 1   ident                499997 non-null  object             
 2   authuser             499997 non-null  object             
 3   datetime             499997 non-null  datetime64[ns, UTC]
 4   method               499997 non-null  object             
 5   url                  499997 non-null  object             
 6   protocol             499997 non-null  object             
 7   status               499997 non-null  Int64              
 8   bytes                499997 non-null  Int64              
 9   referer              499997 non-null  object             
 10  user_agent           499997 non-null  object             
 11  extra                499997 non-null  object             
 12  li